<a href="https://colab.research.google.com/github/srikanth713/Machine_Learning/blob/main/diabetes_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Load Data

First, I'll load the `Diabetes_prediction.csv` file into a pandas DataFrame and display its first 5 rows to understand its structure.

In [ ]:
import pandas as pd

file_path = '/content/Diabetes_prediction.csv'
df = pd.read_csv(file_path)

display(df.head())

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diagnosis
0,2,115.863387,56.410731,24.336736,94.385783,26.455940,0.272682,20.100494,0
1,2,92.490122,70.615520,23.443591,138.652426,23.910167,0.665160,44.912281,0
2,1,88.141469,63.262618,23.404364,149.358082,21.948250,0.676022,48.247873,1
3,2,108.453101,67.793632,20.751580,108.751638,24.209304,0.289636,42.749868,0
4,1,127.849443,94.725685,22.603078,25.269987,32.997477,0.601315,32.797789,0


## 2. Data Preprocessing

I will check for missing values, data types, and prepare the data for the logistic regression model. The 'Outcome' column will be our target variable, and the rest will be features.

In [ ]:
display(df.info())
print('\nMissing values:\n', df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               1000 non-null   int64  
 1   Glucose                   1000 non-null   float64
 2   BloodPressure             1000 non-null   float64
 3   SkinThickness             1000 non-null   float64
 4   Insulin                   1000 non-null   float64
 5   BMI                       1000 non-null   float64
 6   DiabetesPedigreeFunction  1000 non-null   float64
 7   Age                       1000 non-null   float64
 8   Diagnosis                 1000 non-null   int64  
dtypes: float64(7), int64(2)
memory usage: 70.4 KB


None


Missing values:
 Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Diagnosis                   0
dtype: int64


It appears there are no missing values. Next, I'll separate the features (X) from the target variable (y) and split the data into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df.drop('Diagnosis', axis=1)
y = df['Diagnosis']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale numerical features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")

Training set shape: (800, 8)
Testing set shape: (200, 8)


## 3. Build and Train Logistic Regression Model

Now, I'll build a Logistic Regression model using `sklearn` and train it with the scaled training data.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize and train the Logistic Regression model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


## 4. Model Evaluation

I'll evaluate the model's performance on the test set using accuracy score.

In [ ]:
from sklearn.metrics import accuracy_score

# Make predictions on the scaled test set
y_pred = model.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.4f}")

Model Accuracy on Test Set: 0.6850


## 5. Export the Model

Finally, I'll export the trained Logistic Regression model and the `StandardScaler` to a file using `joblib` so it can be used later for predictions.

In [ ]:
import joblib

# Define file paths for saving the model and scaler
model_filename = 'logistic_regression_model.pkl'
scaler_filename = 'standard_scaler.pkl'

# Save the trained model
joblib.dump(model, model_filename)
print(f"Logistic Regression model saved as '{model_filename}'")

# Save the scaler
joblib.dump(scaler, scaler_filename)
print(f"StandardScaler saved as '{scaler_filename}'")

# To load the model later:
# loaded_model = joblib.load(model_filename)
# loaded_scaler = joblib.load(scaler_filename)

Logistic Regression model saved as 'logistic_regression_model.pkl'
StandardScaler saved as 'standard_scaler.pkl'


## 6. Create Flask API for Prediction

Now, I'll create an `app.py` file that sets up a simple Flask API. This API will load the trained Logistic Regression model and the `StandardScaler`, and expose a `/predict` endpoint where you can send new data to get predictions.

Make sure the `logistic_regression_model.pkl` and `standard_scaler.pkl` files are in the same directory as `app.py` when you run it.

In [13]:
app_py_content = """
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

# Load the trained model and scaler
try:
    model = joblib.load('logistic_regression_model.pkl')
    scaler = joblib.load('standard_scaler.pkl')
    print("Model and scaler loaded successfully.")
except Exception as e:
    print(f"Error loading model or scaler: {e}")
    model = None
    scaler = None

@app.route('/')
def home():
    return "Logistic Regression API is running! Send POST requests to /predict."

@app.route('/predict', methods=['POST'])
def predict():
    if model is None or scaler is None:
        return jsonify({'error': 'Model or scaler not loaded'}), 500

    try:
        data = request.get_json(force=True)

        # Ensure input data has the correct feature names and order
        # Based on your training data, the features are:
        # Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age
        feature_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

        # Convert input dictionary to a pandas DataFrame
        # It's important that the order of columns matches the training data
        input_df = pd.DataFrame([data], columns=feature_names)

        # Scale the input features
        scaled_input = scaler.transform(input_df)

        # Make prediction
        prediction = model.predict(scaled_input)
        prediction_proba = model.predict_proba(scaled_input)

        # Return prediction as JSON
        return jsonify({
            'prediction': int(prediction[0]),
            'probability_no_diabetes': prediction_proba[0][0],
            'probability_diabetes': prediction_proba[0][1]
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 400

if __name__ == '__main__':
    # In a production environment, you might use a more robust web server like Gunicorn or uWSGI
    # For local development, debug=True provides useful error messages and auto-reloading
    app.run(host='0.0.0.0', port=5000, debug=True)
"""

with open('app.py', 'w') as f:
    f.write(app_py_content)

print("Flask API script 'app.py' recreated successfully for deployment!")

Flask API script 'app.py' recreated successfully for deployment!


### 7. Create `requirements.txt`

Render needs a `requirements.txt` file to know which Python packages to install for your application. This file lists all the dependencies required for your Flask API to run.

In [14]:
requirements_content = """
Flask==2.3.2
gunicorn==22.0.0 # Gunicorn is a production-ready WSGI HTTP Server
scikit-learn==1.2.2
pandas==1.5.3
joblib==1.2.0
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("Created 'requirements.txt' for deployment.")

Created 'requirements.txt' for deployment.


### 8. Create `render.yaml` (Render Blueprint)

A `render.yaml` file (also known as a Render Blueprint) defines your infrastructure as code. It tells Render how to build and deploy your service, including environment variables, build commands, and start commands.

```
Flask==2.3.2
gunicorn==22.0.0 # Gunicorn is a production-ready WSGI HTTP Server
scikit-learn==1.2.2
pandas==1.5.3
joblib==1.2.0
```

```
Flask==2.3.2
gunicorn==22.0.0 # Gunicorn is a production-ready WSGI HTTP Server
scikit-learn==1.2.2
pandas==1.5.3
joblib==1.2.0
```

In [15]:
render_yaml_content = """
# render.yaml

services:
  - type: web
    name: diabetes-prediction-api
    env: python
    buildCommand: "pip install -r requirements.txt"
    startCommand: "gunicorn app:app"
    # Adjust the port if your Flask app runs on a different one (default is 5000)
    # Render automatically sets the PORT environment variable for your service.
    # Gunicorn respects the PORT environment variable by default.
    # If you need to specify it, use: startCommand: "gunicorn --bind 0.0.0.0:$PORT app:app"
    ports:
      - port: 5000
        name: http
        protocol: HTTP
    numInstances: 1
    plan: free # or 'starter', 'standard', etc.
"""

with open('render.yaml', 'w') as f:
    f.write(render_yaml_content)

print("Created 'render.yaml' for deployment to Render.")

Created 'render.yaml' for deployment to Render.


### Deployment Steps for Render

Now that you have `app.py`, `logistic_regression_model.pkl`, `standard_scaler.pkl`, `requirements.txt`, and `render.yaml`, here's how you would deploy your Flask API to Render:

1.  **Initialize a Git Repository:**
    If you haven't already, initialize a Git repository in your project directory (the same directory where `app.py`, `*.pkl` files, `requirements.txt`, and `render.yaml` are located) and commit all these files.

    ```bash
    git init
    git add .
    git commit -m "Initial commit for Flask API deployment"
    ```

2.  **Create a GitHub/GitLab Repository:**
    Create a new public or private repository on GitHub or GitLab and push your local repository to it.

    ```bash
    git remote add origin <YOUR_REPO_URL>
    git branch -M main
    git push -u origin main
    ```

3.  **Log in to Render:**
    Go to [Render.com](https://render.com/) and log in to your account. If you don't have one, you can sign up for free.

4.  **Create a New Blueprint Instance:**
    *   On your Render Dashboard, click on "New" -> "Blueprint Instance".
    *   Connect your GitHub/GitLab account and select the repository you just pushed your project to.

5.  **Configure Blueprint:**
    *   Render will automatically detect the `render.yaml` file in your repository.
    *   Review the service configuration. The `render.yaml` file defines a `web` service named `diabetes-prediction-api` which will build dependencies from `requirements.txt` and start the app using `gunicorn app:app`.
    *   Click "Apply" to create and deploy your service.

6.  **Monitor Deployment:**
    Render will now clone your repository, install dependencies, and deploy your Flask application. You can monitor the deployment logs from the Render dashboard.

7.  **Access Your API:**
    Once the deployment is successful, Render will provide a public URL for your service. You can use this URL to send requests to your `/predict` endpoint, just like you would with `ngrok` or a local server.

This setup provides a robust and scalable way to host your machine learning model as a web service.

```python
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

# Load the trained model and scaler
try:
    model = joblib.load('logistic_regression_model.pkl')
    scaler = joblib.load('standard_scaler.pkl')
    print("Model and scaler loaded successfully.")
except Exception as e:
    print(f"Error loading model or scaler: {e}")
    model = None
    scaler = None

@app.route('/')
def home():
    return "Logistic Regression API is running! Send POST requests to /predict."

@app.route('/predict', methods=['POST'])
def predict():
    if model is None or scaler is None:
        return jsonify({'error': 'Model or scaler not loaded'}), 500

    try:
        data = request.get_json(force=True)

        # Ensure input data has the correct feature names and order
        # Based on your training data, the features are:
        # Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age
        feature_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

        # Convert input dictionary to a pandas DataFrame
        # It's important that the order of columns matches the training data
        input_df = pd.DataFrame([data], columns=feature_names)

        # Scale the input features
        scaled_input = scaler.transform(input_df)

        # Make prediction
        prediction = model.predict(scaled_input)
        prediction_proba = model.predict_proba(scaled_input)

        # Return prediction as JSON
        return jsonify({
            'prediction': int(prediction[0]),
            'probability_no_diabetes': prediction_proba[0][0],
            'probability_diabetes': prediction_proba[0][1]
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 400

if __name__ == '__main__':
    # In a production environment, you might use a more robust web server like Gunicorn or uWSGI
    # For local development, debug=True provides useful error messages and auto-reloading
    app.run(host='0.0.0.0', port=5000, debug=True)
```

### How to Run the Flask API and Make Predictions

1.  **Run the Flask App:**

    After executing the code cell above, you will have an `app.py` file in your Colab environment. To run it, you would typically open a terminal or command prompt in the directory where `app.py`, `logistic_regression_model.pkl`, and `standard_scaler.pkl` are located, and execute:
    ```bash
    python app.py
    ```
    
    **Note for Colab:** Running a Flask app directly in a Colab notebook cell usually blocks the cell and doesn't expose a public endpoint easily. For local testing, you can download `app.py` and the `.pkl` files and run them on your machine. If you want to expose it from Colab, you would typically use a tool like `ngrok`.

2.  **Make a Prediction (Example using `curl` or Python `requests`):**

    Once the Flask app is running (e.g., on `http://127.0.0.1:5000`), you can send a POST request to the `/predict` endpoint with your new data in JSON format. The feature names in your JSON input must match the ones used during training:

    ```json
    {
        "Pregnancies": 2,
        "Glucose": 115.86,
        "BloodPressure": 56.41,
        "SkinThickness": 24.33,
        "Insulin": 94.38,
        "BMI": 26.45,
        "DiabetesPedigreeFunction": 0.27,
        "Age": 20.10
    }
    ```

    **Using `curl` in your terminal:**
    ```bash
    curl -X POST -H "Content-Type: application/json" \
         -d '{"Pregnancies": 2, "Glucose": 115.86, "BloodPressure": 56.41, "SkinThickness": 24.33, "Insulin": 94.38, "BMI": 26.45, "DiabetesPedigreeFunction": 0.27, "Age": 20.10}' \
         http://127.0.0.1:5000/predict
    ```

    **Using Python `requests`:**
    ```python
    import requests
    import json

    url = 'http://127.0.0.1:5000/predict'
    headers = {'Content-Type': 'application/json'}
    data = {
        "Pregnancies": 2,
        "Glucose": 115.86,
        "BloodPressure": 56.41,
        "SkinThickness": 24.33,
        "Insulin": 94.38,
        "BMI": 26.45,
        "DiabetesPedigreeFunction": 0.27,
        "Age": 20.10
    }

    response = requests.post(url, headers=headers, data=json.dumps(data))
    print(response.json())
    ```

This setup allows you to send new data to your model and receive predictions via an HTTP API.

In [ ]:
!ls -l

total 148
-rw-r--r-- 1 root root   2115 Mar 31 09:22 app.py
-rw-r--r-- 1 root root 133618 Mar 31 09:14 Diabetes_prediction.csv
-rw-r--r-- 1 root root    927 Mar 31 09:17 logistic_regression_model.pkl
drwxr-xr-x 1 root root   4096 Mar 23 13:29 sample_data
-rw-r--r-- 1 root root   1159 Mar 31 09:17 standard_scaler.pkl
